## Testing Embeddings

In [ ]:
from FlagEmbedding import FlagModel 
import os

os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = 'true'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
model = FlagModel('BAAI/bge-base-en-v1.5')


sentences = [
    "That is a happy dog",
    "That is a very happy person",
    "Today is a sunny day",
]

embeddings = model.encode(sentences)

In [ ]:
print(f"Embeddings:\n{embeddings.shape}")

In [ ]:
scores = embeddings @ embeddings.T
print(f"Similiarty scores:\n{scores}")

## Testing File Things

In [4]:
import json

In [ ]:
from pathlib import Path


def read_file_contents(file_path):
    file_contents = ""
    with open(file_path, 'r', encoding='utf-8') as f:
        file_contents += f.read()
    return file_contents


with open('../output/wikipedia-pets/wikipedia-pets/page_index.json', mode='r', encoding='utf-8-sig') as f:
    for entry in json.load(f):
        try: 
            curr_file = entry["text_file"].replace('texts/', '')
            curr_page_id = entry["page_id"]
            open("../data/corpus_cleaned/" + curr_file, 'r', encoding='utf-8')

        except AttributeError as e:
            fail_counter += 1
            print(f"Skipping 404 - {curr_file}")
        

IndentationError: expected an indented block after 'except' statement on line 22 (3052978576.py, line 22)

## Actual Script

In [1]:
import spacy
import csv
import json

In [2]:
def read_file_contents(file_path):
    file_contents = ""
    with open(file_path, 'r', encoding='utf-8') as f:
        file_contents += f.read()
    return file_contents

In [3]:
def smart_text_chunking(text, chunk_size=512):
    nlp = spacy.load("en_core_web_sm")
    nlp.add_pipe('sentencizer')

    sentences = [sent.text for sent in nlp(text).sents]

    for sentence in sentences:
        if len(sentence.split()) > chunk_size:
            raise ValueError(f"Sentence is too long to fit in a chunk: {sentence}")

    # 1. Count words in sentences until >= 512, when >= 512, create chunk then start new chunk. 

    chunks = []

    current_chunk = ""

    for sentence in sentences:
        if len(sentence.split()) + len(current_chunk.split()) <= chunk_size:
            current_chunk += " " + sentence 
        else:
            chunks.append(current_chunk)
            current_chunk = sentence

    return chunks

In [ ]:
REL_CHUNK_INFORMATION_FILE = "../data/embeddings/chunk_information.csv"
REL_PAGE_INDEX_FILE = "../output/wikipedia-pets/wikipedia-pets/page_index.json"

def get_file_chunks(corpus_dir):
    # Matching chunking to embeddings.  
    # - Read the JSON from output/wikipedia_pets/page_index.json. 
    # - Read article file_name, and match it to page_index.json's text_file field without the texts/ prefix. 
    # - Match chunks with files via page_id, so we can track which embeddings belong to which files. 

    #all_article_chunks = []

    # Because script is very slow, we have to track how far we reached in the previous write session before we continue with this one. 
    # - This means first opening the file in read mode and reading the last line to see the page id of the last written file. 
    # - Then saving the last page id and chunk_id, and proceeding from there, rather than writing evrything again. 
    last_done_id = ""
    try: 
        with open(REL_CHUNK_INFORMATION_FILE, mode='r', encoding='utf-8') as f: 
            for line in csv.reader(f): 
                
    except FileNotFoundError as e: 
        print("No chunk data, starting from the beggining.")
        

    # page_id, chunk_id, chunk_text
    chunk_info_file = open(REL_CHUNK_INFORMATION_FILE, mode='w', encoding='utf-8')
    chunk_info_writer = csv.writer(chunk_info_file)

    with open(REL_PAGE_INDEX_FILE, mode='r', encoding='utf-8-sig') as f:
        for entry in json.load(f):
            try: 
                curr_file = entry["text_file"].replace('texts/', '')
                curr_page_id = entry["page_id"]
                open(f"{corpus_dir}{curr_file}" , 'r', encoding='utf-8')
                curr_chunks = smart_text_chunking(read_file_contents(f"{corpus_dir}{curr_file}"))
                if len(curr_chunks) == 1:
                    # page_id, chunk_id, chunk_text
                    chunk_info_writer.writerow([curr_page_id, 0, curr_chunks[0]])
                else:
                    for i, chunk in enumerate(curr_chunks):
                        chunk_info_writer.writerow([curr_page_id, i, chunk])
                # all_article_chunks.extend(curr_chunks)
                
            except AttributeError as e:
                print(f"Skipping 404 - {curr_file}")

            except ValueError as e:
                print (f"Skipping sentence that is too long in {curr_file}")
            
    return all_article_chunks

IndentationError: expected an indented block after 'for' statement on line 15 (1961453264.py, line 18)

In [17]:
get_file_chunks('../data/corpus_cleaned/')

KeyboardInterrupt: 

In [ ]:
source_data = {
    "0", "miro"
    "1", "giro"
    "2", "sviro"
    "3", "diro"
    "4", "biro"
    "5", "ciro"
}

try: 
    with open('../data/example.csv', mode='w', encoding='utf-8') as f: 
        testing_writer = csv.writer(f)
        testing_writer.writerow(["page_id", "chunk_id", "hello_id"])
except FileNotFoundError as e: 
    print("No chunk data, starting from the beggining.")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path


REL_CHUNK_METADATA_FILE = Path("../data/embeddings/chunk_metadata.parquet")
REL_CHUNK_EMBEDDINGS_FILE = Path("../data/embeddings/chunk_embeddings.npy")
REL_PAGE_INDEX_FILE = Path("../output/wikipedia-pets/wikipedia-pets/page_index.json")


def build_chunk_metadata(corpus_dir, page_index_file=REL_PAGE_INDEX_FILE, chunk_size=512):
    corpus_dir = Path(corpus_dir)
    rows = []
    skipped_pages = []
    global_chunk_id = 0

    with open(page_index_file, mode="r", encoding="utf-8-sig") as f:
        for entry in json.load(f):
            text_file = entry.get("text_file")
            if not text_file:
                skipped_pages.append({
                    "page_id": entry.get("page_id"),
                    "title": entry.get("title"),
                    "reason": "missing text_file",
                })
                continue

            curr_file = text_file.replace("texts/", "")
            file_path = corpus_dir / curr_file

            if not file_path.exists():
                skipped_pages.append({
                    "page_id": entry.get("page_id"),
                    "title": entry.get("title"),
                    "file_name": curr_file,
                    "reason": "missing corpus file",
                })
                continue

            try:
                text = read_file_contents(file_path)
                curr_chunks = smart_text_chunking(text, chunk_size=chunk_size)
            except ValueError as e:
                skipped_pages.append({
                    "page_id": entry.get("page_id"),
                    "title": entry.get("title"),
                    "file_name": curr_file,
                    "reason": str(e),
                })
                continue

            for chunk_index, chunk_text in enumerate(curr_chunks):
                chunk_text = chunk_text.strip()
                if not chunk_text:
                    continue

                rows.append({
                    "global_chunk_id": global_chunk_id,
                    "page_id": entry["page_id"],
                    "title": entry.get("title", ""),
                    "file_name": curr_file,
                    "chunk_index": chunk_index,
                    "word_count": len(chunk_text.split()),
                    "char_count": len(chunk_text),
                    "text": chunk_text,
                })
                global_chunk_id += 1

    chunk_metadata = pd.DataFrame(rows)
    skipped_pages = pd.DataFrame(skipped_pages)
    return chunk_metadata, skipped_pages


def save_chunk_metadata_parquet(corpus_dir="../data/corpus_cleaned/", output_file=REL_CHUNK_METADATA_FILE):
    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    chunk_metadata, skipped_pages = build_chunk_metadata(corpus_dir)
    chunk_metadata.to_parquet(output_file, index=False)

    print(f"Saved {len(chunk_metadata)} chunks to {output_file}")
    print(f"Skipped {len(skipped_pages)} pages")
    return chunk_metadata, skipped_pages


def encode_chunks_from_metadata(chunk_metadata, output_file=REL_CHUNK_EMBEDDINGS_FILE):
    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    texts = chunk_metadata.sort_values("global_chunk_id")["text"].tolist()
    embeddings = model.encode(texts)
    np.save(output_file, embeddings)

    print(f"Saved embeddings with shape {embeddings.shape} to {output_file}")
    return embeddings


In [5]:
chunk_metadata, skipped_pages = save_chunk_metadata_parquet("../data/corpus_cleaned/")
# embeddings = encode_chunks_from_metadata(chunk_metadata)# Run manually when ready:
# chunk_metadata, skipped_pages = save_chunk_metadata_parquet("../data/corpus_cleaned/")
# embeddings = encode_chunks_from_metadata(chunk_metadata)

KeyboardInterrupt: 